In [ ]:
# Credit Card Approval Predictor

"""Descripción
Modelo de Machine Learning para predecir automáticamente
si una solicitud de tarjeta de crédito será aprobada o rechazada,
similar a los sistemas usados por bancos comerciales reales."""

In [3]:
import pandas as pd

#se esta corriendo desde la pagina del data set, ya que hubo problemas con la subida a colab

url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/credit-screening/crx.data'
df = pd.read_csv(url, header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,f,g,00202,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,f,g,00043,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,f,g,00280,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,t,g,00100,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,f,s,00120,0,+


In [ ]:
"""vemos columnas 0, 1, 2...15  son  los nombres anonimizados (A1, A2... en el original)
Letras (b, a, u, g, w...) las variables categóricas, esas necesitarán encoding
Números (30.83, 4.46...) las variables continuas, esas necesitarán escalado
Columna 15 con + o -  esa es tu variable objetivo ✅ aprobado/rechazado"""


In [4]:
#exploración de datos


# Tamaño del dataset
print("Forma del dataset:", df.shape)

# Tipos de datos
print("\nTipos de datos:")
print(df.dtypes)

# Valores nulos
print("\nValores nulos por columna:")
print(df.isnull().sum())






Forma del dataset: (690, 16)

Tipos de datos:
0      object
1      object
2     float64
3      object
4      object
5      object
6      object
7     float64
8      object
9      object
10      int64
11     object
12     object
13     object
14      int64
15     object
dtype: object

Valores nulos por columna:
0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
14    0
15    0
dtype: int64


In [5]:
#nulos

"""En total tienes 67 valores faltantes repartidos en 6 columnas. Nota que:

Columna 1 con 12 nulos
Columna 13 con 13 nulos, la más afectada
Las columnas numéricas 2, 7, 10, 14 → sin nulos """




# Reemplazar ? por NaN (nulo real)
import numpy as np

df = df.replace('?', np.nan)

# Ahora sí verificamos
print(df.isnull().sum())

0     12
1     12
2      0
3      6
4      6
5      9
6      9
7      0
8      0
9      0
10     0
11     0
12     0
13    13
14     0
15     0
dtype: int64


In [7]:
#Limpieza

"""La estrategia es:

Columnas categóricas (object) → rellenar con la moda (el valor más frecuente)
Columnas numéricas (float/int) → rellenar con la media"""



# Rellenar nulos - forma correcta para Pandas 3.0
for col in df.columns:
    if df[col].dtype == 'float64' or df[col].dtype == 'int64':
        df[col] = df[col].fillna(df[col].mean())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

# Verificar
print(df.isnull().sum())

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
14    0
15    0
dtype: int64


In [8]:
# encoding — convertir las columnas de texto a números para que el modelo las entienda

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

print(df.head())


#la columna 15 tiene ahora  0= rechazado 1=aprobado



   0    1      2   3   4   5   6     7   8   9   10  11  12  13   14  15
0   1  156  0.000   1   0  12   7  1.25   1   1   1   0   0  68    0   0
1   0  328  4.460   1   0  10   3  3.04   1   1   6   0   0  11  560   0
2   0   89  0.500   1   0  10   3  1.50   1   0   0   0   0  96  824   0
3   1  125  1.540   1   0  12   7  3.75   1   1   5   1   0  31    3   0
4   1   43  5.625   1   0  12   7  1.71   1   0   0   0   2  37    0   0


In [9]:
#predicción

"""690 filas de solicitudes
15 columnas de entrada para X
690 valores para predecir en y"""

# Separar features y variable objetivo
X = df.iloc[:, :-1]  # Todo menos la última columna
y = df.iloc[:, -1]   # Solo la última columna

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (690, 15)
y shape: (690,)


In [10]:
#se divide el dataset en entrenamiento y prueba

#0% de los datos ,el modelo aprende con ellos
#20%  los usamos para probar qué tan bien aprendió

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Entrenamiento: (552, 15)
Prueba: (138, 15)


In [11]:
"""escalado — aquí entra la normalización necesaria para la Regresión Logística:

En X_train usamos fit_transform → aprende la escala Y la aplica
En X_test usamos solo transform → solo aplica la escala que ya aprendió

"""


from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Escalado completo ✅")

Escalado completo ✅


In [12]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

print("Modelo entrenado ✅")

Modelo entrenado ✅


In [13]:

#prueba

from sklearn.metrics import accuracy_score, confusion_matrix

# Predecir
y_pred = modelo.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

# Matriz de confusión
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 82.61%

Matriz de confusión:
[[60 10]
 [14 54]]


60 rechazados  el modelo predijo rechazado correctamente
54 aprobados  el  modelo predijo aprobado correctamente
10 aprobados → predijo rechazado (falso negativo — le negó la tarjeta a alguien que sí calificaba)
14 rechazados → predijo aprobado (falso positivo — le dio tarjeta a alguien que no calificaba).

**El error más caro para el banco es el falso positivo — darle tarjeta a quien no debe. Aquí tienes 14, que es razonable.**


In [14]:

"""De 82% a 87% solo optimizando parámetros!
¿Qué encontró GridSearchCV?

C: 0.1 → controla qué tan estricto es el modelo, 0.1 significa que es más "relajado" y evita overfitting
solver: liblinear → el algoritmo interno que mejor funciona con este dataset"""



from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear']
}

grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5)
grid.fit(X_train, y_train)

print("Mejores parámetros:", grid.best_params_)
print(f"Mejor accuracy: {grid.best_score_:.2%}")

Mejores parámetros: {'C': 0.1, 'solver': 'liblinear'}
Mejor accuracy: 86.95%


In [15]:
"""
¿Por qué el 86.95% del GridSearch no coincide con el 84.06%?
Porque el 86.95% fue medido con validación cruzada (cv=5) — divide el entrenamiento en 5 partes y promedia.
 Es una estimación. El 84.06% es el resultado real contra datos nuevos — ese es el número honesto. """

# Modelo final con mejores parámetros
mejor_modelo = grid.best_estimator_

y_pred_final = mejor_modelo.predict(X_test)

accuracy_final = accuracy_score(y_test, y_pred_final)
print(f"Accuracy final: {accuracy_final:.2%}")
print("\nMatriz de confusión final:")
print(confusion_matrix(y_test, y_pred_final))

Accuracy final: 84.06%

Matriz de confusión final:
[[60 10]
 [12 56]]


In [ ]:

# =============================================================
# CONCLUSIONES APROBACION DE TARJETAS
# =============================================================

# El modelo de Regresión Logística logró predecir aprobaciones
# de tarjetas de crédito con un 84.06% de accuracy sobre datos
# que nunca había visto durante el entrenamiento.

# Proceso seguido:
# 1. El dataset contenía 690 solicitudes con 15 variables anonimizadas
# 2. Se detectaron y limpiaron 67 valores nulos disfrazados como '?'
# 3. Se aplicó LabelEncoder para convertir variables categóricas a números
# 4. Se normalizó con MinMaxScaler, necesario para Regresión Logística
# 5. Se optimizaron hiperparámetros con GridSearchCV (cv=5)
#    - Mejor configuración: C=0.1, solver='liblinear'

# Resultado final en datos de prueba (138 solicitudes):
# - Accuracy: 84.06%
# - Falsos positivos (riesgo para el banco): 12
# - Falsos negativos (clientes perdidos): 10

# El modelo es funcional como prueba de concepto.
# Para producción real se recomienda:
# - Mayor volumen de datos
# - Variables con nombres reales para interpretabilidad
# - Explorar modelos como Random Forest o XGBoost
# =============================================================


